In [ ]:
# 01 · MEDIUM v2 CONFIG
CFG = {
    # 저장 · v1 최고 모델에서 시작, v2 결과는 별도 Release
    'run_name': 'moveboxes_medium_deadline_v2',
    'source_run_name': 'moveboxes_medium_lab_v1',
    'warm_start': True,
    'profile': 'benchmark',
    'github_repository': 'SongYunu/moveBoxes',
    'project_dir': '/content/moveBoxes',
    'project_ref': 'main',
    'output_root': '/content/moveboxes_runs',

    # T4 / Python 3.12 / Colab 2026.07
    'repo_dir': '/content/berlin-marso-hackathon',
    'repo_url': 'https://github.com/marso-robotics/berlin-marso-hackathon.git',
    'repo_commit': '6048f33217f26ae39009a812f53c81171517f393',
    'data_dir': '/content/marso_data',
    'data_source': '/content/moveboxes_data_cache/marso_state_data.zip',
    'download_cache': '/content/moveboxes_data_cache',
    'packages': ['mani-skill==3.0.1', 'sapien==3.0.3', 'diffusers==0.38.0', 'hydra-core', 'omegaconf', 'gymnasium', 'tyro', 'h5py', 'kagglehub', 'tensorboard', 'matplotlib', 'transforms3d', 'imageio[ffmpeg]'],

    # 시연 · 199행동 안에 네 상자가 성공한 실제 시연만 사용
    'teacher_gains': [1.25, 1.75],
    'pilot_episodes': 2,
    'deadline_episodes': 16,
    'deadline_max_attempts': 40,
    'collection_seconds_per_call': 600,
    'pilot_seed_start': 120000,
    'deadline_seed_start': 121000,
    'deadline_noise_std': 0.08,
    'deadline_noise_probability': 0.1,

    # 보정 · 500회마다 평가, 최대 3,000회, 2구간 개선 없으면 종료
    'block_iters': 500,
    'max_blocks': 6,
    'batch_size': 32,
    'lr': 3e-05,
    'amp': True,
    'plateau_blocks': 2,
    'target_accuracy': 0.95,
    'success_streak': 2,

    # 기존 모델 구조 유지 / 집기 접촉 구간은 모든 상자에서 표집
    'seed': 42,
    'num_demos': None,
    'history': 16,
    'chunk_size': 16,
    'width': 128,
    'heads': 4,
    'layers': 2,
    'latent_dim': 16,

    # 개발 / 테스트 / 최종 평가 시드 분리
    'development_episodes': 8,
    'test_episodes': 8,
    'final_episodes': 100,
    'tuning_seed_start': 52000,
    'test_seed_start': 42000,
    'eval_seed_start': 72000,
    'max_episode_steps': {'easy': 200, 'medium': 200, 'hard': 200},

    # 실행 / 출력
    'ensemble_candidates': [4],
    'ensemble_window': 4,
    'temporal_decay': 0.25,
    'gate_threshold': 0.65,
    'stage_threshold': 0.6,
    'console_interval_seconds': 30,
    'team': 'my-team',

}


In [ ]:
USER_CONFIG = dict(CFG)
# 02 · GitHub 코드 불러오기 (데이터·결과를 위해 Drive를 마운트하지 않습니다)
import importlib, os, subprocess, sys
from pathlib import Path

PROJECT = Path(CFG['project_dir'])
URL = 'https://github.com/'+CFG['github_repository']+'.git'
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', URL, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', 'remote', 'get-url', 'origin'], cwd=PROJECT, text=True).strip()
    if remote != URL:
        raise RuntimeError('기존 프로젝트 폴더가 다른 저장소입니다. project_dir를 새 경로로 바꾸세요.')
subprocess.run(['git', 'fetch', '--depth', '1', 'origin', CFG['project_ref']], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=PROJECT, check=True)
CFG['project_commit'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
sys.path.insert(0, str(PROJECT))
# A fresh notebook run should not retain a previously imported project module.
for name in ('marso_experiment', 'marso_train_test', 'next_pick_sampling', 'next_pick_diagnostics',
             'marso_next_pick', 'github_store', 'github_data', 'colab_layout', 'build_modular_notebook',
             'colab_train_test_layout', 'build_train_test_notebook', 'colab_next_pick_layout',
             'build_next_pick_notebook', 'build_github_notebook', 'marso_github'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'))
for name in ('act_v2_model','act_v2_data','act_v2_policy','act_v2_eval','act_v2_experiment','build_act_v2_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
sys.path.insert(0, str(PROJECT/'ver2'/'stages'))
for name in ('stage_schema','stage_model','stage_policy','stage_labels','stage_data','stage_teacher',
             'stage_collect','stage_eval','stage_experiment','build_stage_notebook'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
for name in ('build_medium_notebook','medium_lab'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from build_medium_notebook import CONFIG as MEDIUM_DEFAULTS
CFG = dict(MEDIUM_DEFAULTS, **CFG)
for name in ('build_medium_v2_notebook','medium_v2'):
    if name in sys.modules:
        importlib.reload(sys.modules[name])
from build_medium_v2_notebook import CONFIG as V2_DEFAULTS
PROJECT_COMMIT = CFG['project_commit']
CFG = dict(V2_DEFAULTS, **USER_CONFIG)
CFG['project_commit'] = PROJECT_COMMIT
from medium_v2 import MediumV2, source_bundle
experiment = MediumV2(CFG, source_bundle())
print('사용 코드:', CFG['project_commit'])
print('코드 로드 완료. 03 셀로 연결하거나 다음 실행 셀에서 자동 연결합니다.')


In [ ]:
# 03 · GitHub 인증/복원
experiment.connect()
_ = experiment.report()


In [ ]:
# 04 · 새 런타임마다 환경 설치
experiment.install()


In [ ]:
# 05 · GPU 확인 + 기존 최고 모델 가져오기
experiment.check_runtime()
experiment.prepare()


In [ ]:
# 06 · 빠른 시연 소규모 시험 · 실패하면 학습 진입 차단
_ = experiment.collect_deadline("pilot")


In [ ]:
# 07 · 제한 내 성공 시연 수집 · 시간 예산으로 멈추면 같은 셀 재실행
_ = experiment.collect_deadline("collect")


In [ ]:
# 08 · 보정 학습 + 같은 시드 평가 · 중단하면 같은 셀로 재개
_ = experiment.run_blocks()


In [ ]:
# 09 · 최고 모델 별도 테스트 + 영상 + 행동 기록
experiment.test("medium")
experiment.diagnose("medium")


In [ ]:
# 10 · 구간별 성적 확인
_ = experiment.report()


In [ ]:
# 11 · 선택한 최고 모델 최종 100회 평가 · 부분 점수도 측정
_ = experiment.final_evaluation()


In [ ]:
# 12 · 최종 평가 모델 패키징
_ = experiment.package()
